# 03 – Features and models
Only features that exist live. Compare model families fairly (same data, costs, splits).
Record each comparison in docs/decisions.

Plan: [ADR 0001](../docs/decisions/0001-model-stack.md), protocol: [ADR 0004](../docs/decisions/0004-evaluation-protocol.md), data handling: [ADR 0005](../docs/decisions/0005-data-handling.md).

**Rules of this notebook**
- The **test block is never read**: the candle series is cut before `TEST_START` right after loading.
- Configurations are **selected on the development block**; the validation block only *confirms* the selected one (so its numbers are not inflated by selection on validation). Choosing among the four families on validation is four trials.
- **Every configuration tried is recorded** in a trial register (`docs/trials/03_features_model.csv`) and its count is printed; the more that are tried, the more likely a lucky winner.
- Costs as in R-002 / R-003: taker 3.5 bp from the gateway, funding included (sign and timing assumed), 1x leverage.
- All models use the same features, the same causal standardisation and the same target; batch models are retrained every 168 bars on matured samples only (`walk_forward`), the online model learns from matured samples only (`run_online`).

Needs the cache of notebook 01 (no network) and `scikit-learn` (`pip install -e ".[dev]"`). Set `QUICK = True` for a fast smoke run before the full one.

Status: in progress

In [ ]:
import os
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor

from mlt.data.cache import FrameCache
from mlt.evaluation import bootstrap_sharpe_diff, metrics, sharpe, simulate
from mlt.models.linear import OnlineLinear
from mlt.models.online import run_online
from mlt.models.sklearn_wrap import SklearnForecaster
from mlt.models.walkforward import walk_forward
from mlt.protocol import DATA_END, DATA_START, TEST_START, block_mask, freeze, require_window

warnings.filterwarnings("ignore")           # scikit-learn convergence chatter
QUICK = False                               # True: one horizon, retrain every 672 bars (smoke run)
INSTRUMENT, INTERVAL = "BTC_USDC-PERPETUAL", "1h"
FEE, SLIP = 0.00035, 0.0                    # gateway taker, no extra slippage (as R-002 / R-003)

cache = FrameCache("../data")
raw = cache.read("deribit", INSTRUMENT, "candles", INTERVAL)
require_window(raw, "candle cache")
candles = freeze(raw)
candles = candles.loc[candles.index < TEST_START]          # the test block is never read here
funding = freeze(cache.read("deribit", INSTRUMENT, "funding"))["interest_1h"]
funding = funding.loc[funding.index < TEST_START]
close = candles["close"]
print("frozen window:", DATA_START, "->", DATA_END)
print("used here:", close.index[0], "->", close.index[-1], f"({len(close)} bars; test block cut off)")

## 1. Features (live-only) and a causality check
Log returns over several windows, realised volatility, volume relative to its recent average, candle range, and the hourly funding rate known at the decision bar (lagged one bar; missing funding counts as zero, ADR 0005). Features are standardised with **expanding statistics shifted by one bar**, so a row never sees its own or later data. The check below replaces everything after a cut-off with garbage and asserts that earlier rows do not change.

In [ ]:
def make_features(c, h, l, v, fund):
    lr = np.log(c).diff()
    f = {f"ret_{w}": np.log(c / c.shift(w)) for w in (1, 4, 12, 24, 72, 168)}
    f["vol_24"] = lr.rolling(24).std()
    f["vol_168"] = lr.rolling(168).std()
    lv = np.log1p(v)
    f["volume_z"] = (lv - lv.rolling(168).mean()) / lv.rolling(168).std()
    rng = (h - l) / c
    f["range"] = rng
    f["range_24"] = rng.rolling(24).mean()
    f["funding_lag1"] = fund.reindex(c.index).shift(1).fillna(0.0)
    return pd.DataFrame(f).replace([np.inf, -np.inf], np.nan)


def causal_z(F, min_periods=500, clip=5.0):
    mu = F.expanding(min_periods=min_periods).mean().shift(1)
    sd = F.expanding(min_periods=min_periods).std().shift(1)
    return ((F - mu) / sd).clip(-clip, clip)


def build(c_df, fund):
    F = make_features(c_df["close"], c_df["high"], c_df["low"], c_df["volume"], fund)
    return F, causal_z(F)


F, Z = build(candles, funding)

# causality check: garbage after the cut must not change any earlier row
CUT = 6000
rng_ = np.random.default_rng(0)
junk = candles.copy()
for col in ("open", "high", "low", "close", "volume"):
    junk.loc[junk.index[CUT + 1:], col] *= rng_.uniform(0.5, 2.0, size=len(junk) - CUT - 1)
junk_f = funding.copy()
junk_f.loc[junk_f.index > candles.index[CUT]] = rng_.normal(0, 1e-3, size=int((junk_f.index > candles.index[CUT]).sum()))
Fj, Zj = build(junk, junk_f)
assert np.allclose(F.iloc[: CUT + 1], Fj.iloc[: CUT + 1], equal_nan=True), "features look ahead"
assert np.allclose(Z.iloc[: CUT + 1], Zj.iloc[: CUT + 1], equal_nan=True), "standardisation looks ahead"
print("causality check passed;", F.shape[1], "features:", list(F.columns))
Xz = Z.to_numpy()

## 2. Reference bars on the same bars
Flat, buy-and-hold, the previous linear model (its own features and parameters, as in R-003) and two **no-ML trend rules** (sign of the 24 h or 72 h return). Evaluated from the same first bar as notebook 02 (warm-up 287 + burn-in 2,000), on the development and validation blocks.

In [ ]:
FIRST_PRED = 287 + 2000                     # first evaluated bar of notebook 02
EV = close.index[FIRST_PRED:]
DEV = EV[block_mask(EV, "development")]
VAL = EV[block_mask(EV, "validation")]
print("development:", DEV[0], "->", DEV[-1], len(DEV), "bars | validation:", VAL[0], "->", VAL[-1], len(VAL), "bars")


def positions(pred, thr, allow_shorts=True):
    pos = pd.Series(0.0, index=pred.index)
    pos[pred > thr] = 1.0
    if allow_shorts:
        pos[pred < -thr] = -1.0
    return pos


def run_net(target, window):
    return simulate(close.loc[window], target.loc[window], FEE, SLIP, funding_1h=funding)


# previous linear model exactly as in notebook 02 (windows 125-250, horizon 37, lr 0.00325, L2 0.425)
WINDOWS, HORIZON_OLD = [125, 150, 175, 200, 225, 250], 37
Xold = np.column_stack([np.log(close / close.shift(w)).to_numpy() for w in WINDOWS])
yold = np.log(close.shift(-HORIZON_OLD) / close).to_numpy()
pold = pd.Series(np.clip(run_online(OnlineLinear(6, 0.00325, 0.425), Xold, yold, HORIZON_OLD), -0.2, 0.2), index=close.index)

REFS = {
    "flat": pd.Series(0.0, index=close.index),
    "buy-and-hold": pd.Series(1.0, index=close.index),
    "previous linear model": positions(pold, FEE + SLIP),
    "trend rule 24h": np.sign(F["ret_24"]).fillna(0.0),
    "trend rule 72h": np.sign(F["ret_72"]).fillna(0.0),
}
rows, REF_NET = {}, {}
for name, tgt in REFS.items():
    for blk, win in (("development", DEV), ("validation", VAL)):
        o = run_net(tgt, win)
        m = metrics(o["net"], o["held"])
        rows[(name, blk)] = m
        REF_NET[(name, blk)] = o["net"]
ref_table = pd.DataFrame(rows).T[["sharpe", "total_return", "max_drawdown", "trades"]]
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
ref_table

## 3. Candidate configurations
Four families behind the same `Forecaster` contract. Horizons are the target horizon in hours; `MULTS` multiply the cost to give the position threshold (the app's rule). The grids are small on purpose: every cell is a trial and is counted.

In [ ]:
HORIZONS = (24,) if QUICK else (12, 24, 48)
RETRAIN = 672 if QUICK else 168
MULTS = (1, 4, 8)
GRID = {
    "online_linear": [dict(lr=0.003, l2=0.1), dict(lr=0.01, l2=0.1)],
    "ridge": [dict(alpha=10.0), dict(alpha=1000.0)],
    "hist_gb": [dict(max_depth=3, learning_rate=0.05, max_iter=150), dict(max_depth=2, learning_rate=0.05, max_iter=150)],
    "mlp": [dict(hidden=(16,), alpha=1e-2), dict(hidden=(8,), alpha=1e-1)],
}


def factory(fam, hp):
    if fam == "ridge":
        return lambda: SklearnForecaster(lambda: Ridge(alpha=hp["alpha"]))
    if fam == "hist_gb":
        return lambda: SklearnForecaster(lambda: HistGradientBoostingRegressor(
            max_depth=hp["max_depth"], learning_rate=hp["learning_rate"], max_iter=hp["max_iter"],
            l2_regularization=1.0, early_stopping=False, random_state=0))
    if fam == "mlp":
        return lambda: SklearnForecaster(lambda: MLPRegressor(
            hidden_layer_sizes=hp["hidden"], alpha=hp["alpha"], max_iter=200,
            early_stopping=False, random_state=0))
    raise ValueError(fam)


PREDS = {}
t_all = time.time()
for H in HORIZONS:
    y = np.log(close.shift(-H) / close).to_numpy()          # known only H bars later: handled by the protocols
    for fam, hps in GRID.items():
        t0 = time.time()
        for i, hp in enumerate(hps):
            if fam == "online_linear":
                p = run_online(OnlineLinear(Xz.shape[1], **hp), Xz, y, H)
            else:
                p = walk_forward(factory(fam, hp), Xz, y, H, FIRST_PRED, retrain_every=RETRAIN, min_train=500)
            PREDS[(fam, i, H)] = pd.Series(np.clip(p, -0.2, 0.2), index=close.index)
        print(f"H={H:>2}  {fam:<14} {len(hps)} configs  {time.time() - t0:5.1f}s")
print(f"predictions done in {time.time() - t_all:.0f}s")

## 4. Trial register and selection
Every (family, parameters, horizon, threshold multiplier) is evaluated on both blocks and written to the register. **Selection per family uses the development Sharpe only.**

In [ ]:
rows = []
for (fam, i, H), pr in PREDS.items():
    for mult in MULTS:
        pos = positions(pr, mult * (FEE + SLIP))
        for blk, win in (("development", DEV), ("validation", VAL)):
            o = run_net(pos, win)
            m = metrics(o["net"], o["held"])
            rows.append(dict(family=fam, hp=i, hp_params=str(GRID[fam][i]), horizon=H, mult=mult, block=blk,
                             sharpe=m["sharpe"], total_return=m["total_return"],
                             max_drawdown=m["max_drawdown"], trades=m["trades"], exposure=m["exposure"]))
TRIALS = pd.DataFrame(rows)
os.makedirs("../docs/trials", exist_ok=True)
TRIALS.to_csv("../docs/trials/03_features_model.csv", index=False)
n_trials = TRIALS[["family", "hp", "horizon", "mult"]].drop_duplicates().shape[0]
print("configurations tried:", n_trials, "(register: docs/trials/03_features_model.csv)")

key = ["family", "hp", "horizon", "mult"]
dev = TRIALS[TRIALS.block == "development"].set_index(key)
val = TRIALS[TRIALS.block == "validation"].set_index(key)
BEST = {}
for fam in GRID:
    d = dev.loc[fam]["sharpe"].fillna(-np.inf)
    BEST[fam] = (fam,) + d.idxmax()
summary = pd.DataFrame({
    fam: dict(config=f"hp={k[1]} H={k[2]} mult={k[3]} {GRID[fam][k[1]]}",
              dev_sharpe=dev.loc[k, "sharpe"], val_sharpe=val.loc[k, "sharpe"],
              val_return=val.loc[k, "total_return"], val_drawdown=val.loc[k, "max_drawdown"],
              val_trades=val.loc[k, "trades"])
    for fam, k in BEST.items()}).T
summary

## 5. Do the selected configurations beat the references on validation?
Paired block bootstrap (blocks of one week, 1,000 resamples) of the Sharpe **difference** on the validation bars. The interval measures sampling noise only. A positive difference whose interval excludes zero would be a first sign, not a result: the acceptance rule is fixed in notebook 04 before the test block is read.

In [ ]:
def val_net(k):
    fam, i, H, mult = k
    return run_net(positions(PREDS[(fam, i, H)], mult * (FEE + SLIP)), VAL)["net"]


# the better trend rule is chosen on development
trend_dev = {n: sharpe(REF_NET[(n, "development")]) for n in ("trend rule 24h", "trend rule 72h")}
best_trend = max(trend_dev, key=trend_dev.get)
print("trend rule chosen on development:", best_trend)
out = {}
for fam, k in BEST.items():
    net = val_net(k)
    for ref in ("previous linear model", best_trend, "buy-and-hold"):
        r = bootstrap_sharpe_diff(net, REF_NET[(ref, "validation")], block=168, n_boot=1000, seed=0)
        out[(fam, "vs " + ref)] = r
pd.DataFrame(out).T[["diff", "lo", "hi", "p_not_above_zero"]]

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for fam, k in BEST.items():
    ax.plot((1 + val_net(k)).cumprod(), label=f"{fam} (H={k[2]}, mult={k[3]})")
for ref in ("previous linear model", best_trend, "buy-and-hold"):
    ax.plot((1 + REF_NET[(ref, "validation")]).cumprod(), "--", label=ref)
ax.set_title("Validation block: equity after costs and funding"); ax.legend(fontsize=8); plt.tight_layout()

## Findings (fill in after running; only numbers actually printed above)
- Configurations tried (trial register):
- Reference bars on development and validation (flat, buy-and-hold, previous linear model, the two trend rules):
- Selected configuration per family (chosen on development) and its development / validation Sharpe, return, drawdown, trades:
- Differences against the previous linear model, the better trend rule and buy-and-hold on validation, with the bootstrap intervals:
- Does any family clearly beat the references, or does a no-ML rule do as well?
- Conclusion for ADR 0001 (which families stay, which go) and what is still open:

When this notebook is complete: set `Status: done`, add an `R-004` entry to `docs/RESEARCH_LOG.md` with the numbers and commit `docs/trials/03_features_model.csv`.